In [ ]:
import os
import glob
from PIL import Image

# Prevent DecompressionBombError when Kaggle tries to open massive SAR TIFF files
Image.MAX_IMAGE_PIXELS = None

def process_sar_images(input_tiff_dir, output_dataset_dir, patch_size=256, downscale_factor=4):
    """
    Slices massive TIFF images into HR patches and generates downscaled LR pairs.
    """
    print("⏳ Initializing pre-processing pipeline...")

    # 1. Create output directories exactly as SRGAN expects them
    hr_dir = os.path.join(output_dataset_dir, 'train_HR')
    lr_dir = os.path.join(output_dataset_dir, 'train_LR')
    os.makedirs(hr_dir, exist_ok=True)
    os.makedirs(lr_dir, exist_ok=True)

    # 2. Find all TIFF files in the input folder
    tiff_files = glob.glob(os.path.join(input_tiff_dir, '*.tiff')) + \
                 glob.glob(os.path.join(input_tiff_dir, '*.tif'))

    if not tiff_files:
        print(f"❌ No .tiff files found in {input_tiff_dir}")
        return

    patch_count = 0

    # 3. Process each TIFF file
    for tiff_path in tiff_files:
        filename = os.path.basename(tiff_path).split('.')[0]
        print(f"  -> Slicing {filename}...")

        try:
            # Open the raw SAR image and ensure standard color formatting
            img = Image.open(tiff_path).convert('RGB')
            width, height = img.size

            # Slide a 256x256 window across the image grid
            for top in range(0, height - patch_size + 1, patch_size):
                for left in range(0, width - patch_size + 1, patch_size):
                    right = left + patch_size
                    bottom = top + patch_size

                    # Extract the High-Res Patch (256x256)
                    hr_patch = img.crop((left, top, right, bottom))

                    # Generate the Low-Res Pair (64x64) using Bicubic Interpolation
                    lr_size = patch_size // downscale_factor
                    lr_patch = hr_patch.resize((lr_size, lr_size), Image.Resampling.BICUBIC)

                    # Define standardized filenames
                    patch_name = f"{filename}_patch_{patch_count:05d}.png"
                    hr_save_path = os.path.join(hr_dir, patch_name)
                    lr_save_path = os.path.join(lr_dir, patch_name)

                    # Save both pairs as PNGs
                    hr_patch.save(hr_save_path, 'PNG')
                    lr_patch.save(lr_save_path, 'PNG')

                    patch_count += 1

        except Exception as e:
            print(f"⚠️ Error processing {filename}: {e}")

    print(f"✅ Pre-processing complete! Successfully generated {patch_count} perfectly matched HR/LR pairs.")

# ==========================================
# EXECUTION BLOCK
# ==========================================
# Update this path to where your raw .tiff images are uploaded in Kaggle
INPUT_DIR = "/kaggle/input/your-raw-tiff-dataset-folder"

# This will save the processed patches directly into the Kaggle working directory
OUTPUT_DIR = "/kaggle/working/SAR_Dataset"

process_sar_images(INPUT_DIR, OUTPUT_DIR)

### Do a sanity check of those images generated

In [ ]:
import os
import random
import matplotlib.pyplot as plt
from PIL import Image

def verify_dataset(dataset_dir, num_samples=3):
    """
    Randomly selects and displays pairs of HR and LR images to verify
    the pre-processing pipeline worked correctly.
    """
    hr_dir = os.path.join(dataset_dir, 'train_HR')
    lr_dir = os.path.join(dataset_dir, 'train_LR')

    # Get all the generated filenames
    try:
        all_files = os.listdir(hr_dir)
        if not all_files:
            print("❌ No images found in the HR directory.")
            return
    except FileNotFoundError:
        print(f"❌ Could not find directory: {hr_dir}")
        return

    # Pick a few random files to inspect
    sample_files = random.sample(all_files, min(num_samples, len(all_files)))

    print(f"🔍 Inspecting {len(sample_files)} random pairs from the dataset...")

    # Set up the matplotlib plot
    fig, axes = plt.subplots(len(sample_files), 2, figsize=(10, 5 * len(sample_files)))
    if len(sample_files) == 1:
        axes = [axes] # Handle the case where we only check 1 image

    for i, filename in enumerate(sample_files):
        hr_path = os.path.join(hr_dir, filename)
        lr_path = os.path.join(lr_dir, filename)

        if not os.path.exists(lr_path):
            print(f"⚠️ Warning: Missing matching LR file for {filename}")
            continue

        # Open the images
        hr_img = Image.open(hr_path)
        lr_img = Image.open(lr_path)

        # Plot High-Res
        axes[i][0].imshow(hr_img)
        axes[i][0].set_title(f"High-Res (Ground Truth)\n{hr_img.size[0]}x{hr_img.size[1]}", fontsize=12)
        axes[i][0].axis('off')

        # Plot Low-Res
        axes[i][1].imshow(lr_img)
        axes[i][1].set_title(f"Low-Res (Bicubic Downscaled)\n{lr_img.size[0]}x{lr_img.size[1]}", fontsize=12)
        axes[i][1].axis('off')

    plt.tight_layout()
    plt.show()

# ==========================================
# EXECUTION BLOCK
# ==========================================
# Point this to where you just saved your patches
DATASET_DIR = "/kaggle/working/SAR_Dataset"

verify_dataset(DATASET_DIR, num_samples=4)